# Section1

In [ ]:
# ── Step 0: Cleanup ───────────────────────────────────────────────────────────
import os
import shutil

# Remove any leftover folders from previous crashed sessions
for folder in ['asl_alphabet_train', 'asl_alphabet_test', 'asl_balanced']:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Cleaned up old {folder}")

# ── Step 1: Unzip the dataset ─────────────────────────────────────────────────
!unzip -q 'archive (1).zip'
!echo "Unzip done."

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json
import os
import shutil
import random
import gc

# ── Config ────────────────────────────────────────────────────────────────────
IMAGE_SIZE       = (160, 160)        # Compromise: bigger than 128 and smaller than 228 which crasehed, fits in RAM
BATCH_SIZE       = 32
EPOCHS_FROZEN    = 40                # Phase 1: frozen backbone
EPOCHS_FINETUNE  = 20                # Phase 2: fine-tune top layers
DATASET_DIR      = 'asl_alphabet_train/asl_alphabet_train'
IMAGES_PER_CLASS = 2000
MODEL_NAME       = 'asl_model3_v2.keras'
CLASSES_NAME     = 'class_names3.json'
# ─────────────────────────────────────────────────────────────────────────────

if not os.path.exists(DATASET_DIR):
    print("Expected directory not found. Available directories:")
    os.system('find . -maxdepth 4 -type d')
    raise Exception("Update DATASET_DIR to match the correct path shown above")

print(f"Dataset found at: {DATASET_DIR}")

# ── Remove unwanted classes ───────────────────────────────────────────────────
for remove_class in ['nothing', 'del']:
    class_path = os.path.join(DATASET_DIR, remove_class)
    if os.path.exists(class_path):
        shutil.rmtree(class_path)
        print(f"Removed '{remove_class}' class")

# ── Trim dataset in place (no copying) ───────────────────────────────────────
print(f"\nTrimming to {IMAGES_PER_CLASS} images per class (deleting excess)...")
for cls in sorted(os.listdir(DATASET_DIR)):
    cls_path = os.path.join(DATASET_DIR, cls)
    if not os.path.isdir(cls_path):
        continue
    images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if len(images) > IMAGES_PER_CLASS:
        random.seed(42)
        random.shuffle(images)
        to_delete = images[IMAGES_PER_CLASS:]
        for img in to_delete:
            os.remove(os.path.join(cls_path, img))
        print(f"  {cls}: kept {IMAGES_PER_CLASS}, deleted {len(to_delete)}")
    else:
        print(f"  {cls}: {len(images)} images (no trim needed)")

gc.collect()

# ── Step 2: Load datasets directly ───────────────────────────────────────────
full_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    validation_split=0.2,
    subset='both',
    seed=42
)

train_dataset, validation_dataset = full_dataset

class_names = train_dataset.class_names
NUM_CLASSES  = len(class_names)
print(f"\nClasses ({NUM_CLASSES}): {class_names}")

with open(CLASSES_NAME, 'w') as f:
    json.dump(class_names, f)
print(f"Saved {CLASSES_NAME}")

# ── Step 3: Pipeline ──────────────────────────────────────────────────────────
train_dataset = (
    train_dataset
    .shuffle(500)
    .prefetch(tf.data.AUTOTUNE)
)

validation_dataset = (
    validation_dataset
    .prefetch(tf.data.AUTOTUNE)
)

# ── Step 4: Augmentation layers ───────────────────────────────────────────────
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.2),
    layers.RandomBrightness(0.3),
    layers.RandomContrast(0.3),
    layers.RandomTranslation(0.1, 0.1),
], name="augmentation")

# ── Step 5: MobileNetV2 model ─────────────────────────────────────────────────
base_model = keras.applications.MobileNetV2(
    input_shape=IMAGE_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = keras.Input(shape=IMAGE_SIZE + (3,))
x = data_augmentation(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ── Step 6: Phase 1 — Train with frozen backbone ─────────────────────────────
print("\n=== Phase 1: Training (backbone frozen) ===")
history = model.fit(
    train_dataset,
    epochs=EPOCHS_FROZEN,
    validation_data=validation_dataset,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1),
        keras.callbacks.ModelCheckpoint(
            MODEL_NAME,
            save_best_only=True,
            monitor='val_accuracy',
            verbose=1
        )
    ],
    verbose=1
)

results = model.evaluate(validation_dataset)
print(f"\nPhase 1 Val Loss: {results[0]:.4f} | Val Accuracy: {results[1]:.4f}")

# ── Step 7: Phase 2 — Fine-tune top layers of backbone ───────────────────────
print("\n=== Phase 2: Fine-tuning top 30 layers ===")
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
frozen_count    = sum(1 for l in base_model.layers if not l.trainable)
print(f"Backbone layers — trainable: {trainable_count}, frozen: {frozen_count}")

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),   # 100x lower LR
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = model.fit(
    train_dataset,
    epochs=EPOCHS_FINETUNE,
    validation_data=validation_dataset,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2, verbose=1),
        keras.callbacks.ModelCheckpoint(
            MODEL_NAME,
            save_best_only=True,
            monitor='val_accuracy',
            verbose=1
        )
    ],
    verbose=1
)

# ── Step 8: Final evaluation ──────────────────────────────────────────────────
results = model.evaluate(validation_dataset)
print(f"\nFinal Val Loss: {results[0]:.4f} | Final Val Accuracy: {results[1]:.4f}")

# ── Step 9: Save ──────────────────────────────────────────────────────────────
model.save(MODEL_NAME)
print(f"\nSaved {MODEL_NAME}")
print(f"Saved {CLASSES_NAME}")
print("Download both files manually from the Colab files panel on the left.")

Cleaned up old asl_alphabet_train
Cleaned up old asl_alphabet_test
Unzip done.
Dataset found at: asl_alphabet_train/asl_alphabet_train
Removed 'nothing' class
Removed 'del' class

Trimming to 2000 images per class (deleting excess)...
  A: kept 2000, deleted 1000
  B: kept 2000, deleted 1000
  C: kept 2000, deleted 1000
  D: kept 2000, deleted 1000
  E: kept 2000, deleted 1000
  F: kept 2000, deleted 1000
  G: kept 2000, deleted 1000
  H: kept 2000, deleted 1000
  I: kept 2000, deleted 1000
  J: kept 2000, deleted 1000
  K: kept 2000, deleted 1000
  L: kept 2000, deleted 1000
  M: kept 2000, deleted 1000
  N: kept 2000, deleted 1000
  O: kept 2000, deleted 1000
  P: kept 2000, deleted 1000
  Q: kept 2000, deleted 1000
  R: kept 2000, deleted 1000
  S: kept 2000, deleted 1000
  T: kept 2000, deleted 1000
  U: kept 2000, deleted 1000
  V: kept 2000, deleted 1000
  W: kept 2000, deleted 1000
  X: kept 2000, deleted 1000
  Y: kept 2000, deleted 1000
  Z: kept 2000, deleted 1000
  space: ke

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 27)             │         6,939 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,052,123 (11.64 MB)

 Trainable params: 794,139 (3.03 MB)

 Non-trainable params: 2,257,984 (8.61 MB)


=== Phase 1: Training (backbone frozen) ===
Epoch 1/40
1350/1350 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.3517 - loss: 2.1909
Epoch 1: val_accuracy improved from -inf to 0.79426, saving model to asl_model3_v2.keras
1350/1350 ━━━━━━━━━━━━━━━━━━━━ 98s 52ms/step - accuracy: 0.3518 - loss: 2.1905 - val_accuracy: 0.7943 - val_loss: 0.6235 - learning_rate: 0.0010
Epoch 2/40
1350/1350 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.6314 - loss: 1.1482
Epoch 2: val_accuracy improved from 0.79426 to 0.84093, saving model to asl_model3_v2.keras
1350/1350 ━━━━━━━━━━━━━━━━━━━━ 81s 49ms/step - accuracy: 0.6314 - loss: 1.1481 - val_accuracy: 0.8409 - val_loss: 0.4853 - learning_rate: 0.0010
Epoch 3/40
1349/1350 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.6900 - loss: 0.9656
Epoch 3: val_accuracy improved from 0.84093 to 0.84454, saving model to asl_model3_v2.keras
1350/1350 ━━━━━━━━━━━━━━━━━━━━ 78s 49ms/step - accuracy: 0.6900 - loss: 0.9656 - val_accuracy: 0.8445 - val_loss: 0.4500 - le